In [88]:
import pandas as pd

df = pd.read_csv("data/cleaned/04_shared/master_cleaned.csv")

In [89]:
print(df.columns)

Index(['city', 'state', 'latitude', 'longitude', 'datetime', 'month',
       'day_name', 'is_weekend', 'season', 'time_of_day', 'humidity_percent',
       'dew_point_c', 'wind_gusts_kmh', 'precipitation_mm', 'is_raining',
       'heavy_rain', 'pressure_msl_hpa', 'cloud_cover_percent', 'pm2_5_ugm3',
       'pm10_ugm3', 'co_ugm3', 'no2_ugm3', 'so2_ugm3', 'o3_ugm3', 'dust_ugm3',
       'aod', 'us_aqi', 'aqi_category', 'pm25_category_india',
       'festival_period', 'crop_burning_season', 'hour', 'day_of_week'],
      dtype='str')


In [90]:
df['AQI'] = df['us_aqi']

In [91]:
df = df.sort_values(['city', 'datetime'])

In [92]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['city_enc'] = le.fit_transform(df['city'])

In [93]:
import joblib
joblib.dump(le, "models/city_encoder.pkl")

['models/city_encoder.pkl']

### STEP 4 — Create lag features

In [94]:
for lag in [1, 6, 24]:
    df[f'AQI_lag_{lag}'] = df.groupby('city')['AQI'].shift(lag)

### STEP 5 — Create future targets

In [95]:
df['AQI_next_6']  = df.groupby('city')['AQI'].shift(-6)
df['AQI_next_12'] = df.groupby('city')['AQI'].shift(-12)
df['AQI_next_24'] = df.groupby('city')['AQI'].shift(-24)

In [96]:
df[['city','datetime','AQI','AQI_lag_1','AQI_lag_6','AQI_lag_24']].head(40)

,city,datetime,AQI,AQI_lag_1,AQI_lag_6,AQI_lag_24
0,agartala,2022-08-05 00:00:00,289.0,NaN,NaN,NaN
1,agartala,2022-08-05 01:00:00,289.0,289.0,NaN,NaN
2,agartala,2022-08-05 02:00:00,54.0,289.0,NaN,NaN
3,agartala,2022-08-05 03:00:00,54.0,54.0,NaN,NaN
4,agartala,2022-08-05 04:00:00,54.0,54.0,NaN,NaN
5,agartala,2022-08-05 05:00:00,54.0,54.0,NaN,NaN
6,agartala,2022-08-05 06:00:00,55.0,54.0,289.0,NaN
7,agartala,2022-08-05 07:00:00,55.0,55.0,289.0,NaN
8,agartala,2022-08-05 08:00:00,55.0,55.0,54.0,NaN
9,agartala,2022-08-05 09:00:00,54.0,55.0,54.0,NaN


### STEP 6 — Drop unwanted columns

In [97]:
df = df.drop(columns=[
    'city','state','latitude','longitude','datetime',
    'day_name','season','time_of_day',
    'dust_ugm3','aod',
    'aqi_category','pm25_category_india',
    'festival_period','crop_burning_season',
    'is_raining','heavy_rain'
])

In [99]:
df['is_weekend'] = df['is_weekend'].astype(str).map({
    'True': 1,
    'False': 0
}).fillna(0).astype(int)

### STEP 7 — Select final features

In [100]:
FEATURES = [
    # pollutants
    'pm2_5_ugm3','pm10_ugm3','co_ugm3','no2_ugm3','so2_ugm3','o3_ugm3',

    # time
    'hour','day_of_week','month','is_weekend',

    # location
    'city_enc',

    # lag features
    'AQI_lag_1','AQI_lag_6','AQI_lag_24'
]

joblib.dump(FEATURES, "models/features.pkl")

['models/features.pkl']

### STEP 8 — Drop NaNs 

In [101]:
df = df.dropna()

### STEP 9 — Define X and y

In [102]:
X = df[FEATURES]

y_6  = df['AQI_next_6']
y_12 = df['AQI_next_12']
y_24 = df['AQI_next_24']

### STEP 10 — Train-test split

In [103]:
split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y6_train, y6_test = y_6.iloc[:split], y_6.iloc[split:]
y12_train, y12_test = y_12.iloc[:split], y_12.iloc[split:]
y24_train, y24_test = y_24.iloc[:split], y_24.iloc[split:]

### STEP 11 - Train 3 RandomForest Models

In [121]:
from sklearn.ensemble import RandomForestRegressor

rf_models = {}

for y_train, label in [
    (y6_train, "6h"),
    (y12_train, "12h"),
    (y24_train, "24h")
]:
    print(f"\nTraining RF for {label}")
    
    rf = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    rf.fit(X_train, y_train)
    
    rf_models[label] = rf


Training RF for 6h


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   19.7s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  2.1min finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.



Training RF for 12h


[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   19.2s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  2.1min finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.



Training RF for 24h


[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   20.6s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  2.2min finished


### STEP 11.1 - Evaluations for the Random Forest Regressor

In [122]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

def evaluate_model(model, X_test, y_test, name):
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    print(f"{name}")
    print("R2:", r2)
    print("RMSE:", rmse)
    print()

### STEP 11 - Train 3 XGBoost Models

In [ ]:
pd.DataFrame({
    "Actual": y6_test.values[:20],
    "Predicted": model_6.predict(X_test)[:20]
})

,Actual,Predicted
0,125.0,145.416885
1,121.0,131.141159
2,118.0,123.089966
3,111.0,116.330116
4,108.0,118.194908
5,107.0,116.225098
6,107.0,115.723663
7,107.0,113.070946
8,107.0,111.170128
9,108.0,108.398315


In [104]:
!pip install xgboost


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [105]:
from xgboost import XGBRegressor

model_6  = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
model_12 = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
model_24 = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)

model_6.fit(X_train, y6_train)
model_12.fit(X_train, y12_train)
model_24.fit(X_train, y24_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

### STEP 12 -- EVALS

In [109]:
!pip install -U scikit-learn


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [111]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def evaluate(y_true, y_pred, name):
    print(f"==== {name} ====")
    print("R2   :", r2_score(y_true, y_pred))
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    
    print("RMSE :", rmse)
    print()

In [112]:
evaluate(y6_test, model_6.predict(X_test), "6h")
evaluate(y12_test, model_12.predict(X_test), "12h")
evaluate(y24_test, model_24.predict(X_test), "24h")

==== 6h ====
R2   : 0.970813164862302
RMSE : 6.744911115104681

==== 12h ====
R2   : 0.9043127591234664
RMSE : 12.213651545519935

==== 24h ====
R2   : 0.7745511813432562
RMSE : 18.75464663062243



### Verifying Predictions

In [113]:
pd.DataFrame({
    "Actual": y6_test.values[:20],
    "Predicted": model_6.predict(X_test)[:20]
})

,Actual,Predicted
0,125.0,145.416885
1,121.0,131.141159
2,118.0,123.089966
3,111.0,116.330116
4,108.0,118.194908
5,107.0,116.225098
6,107.0,115.723663
7,107.0,113.070946
8,107.0,111.170128
9,108.0,108.398315


In [117]:
joblib.dump(model_6,  "models/xgb_6h.pkl")
joblib.dump(model_12, "models/xgb_12h.pkl")
joblib.dump(model_24, "models/xgb_24h.pkl")
joblib.dump(FEATURES, "models/features.pkl")
joblib.dump(le, "models/city_encoder.pkl")

['models/city_encoder.pkl']

### QUICK TEST

In [118]:
m6 = joblib.load("models/xgb_6h.pkl")

sample = X_test.iloc[0].values.reshape(1, -1)
print(m6.predict(sample))

[145.41689]


In [ ]:
pd.DataFrame({
    "Actual": y6_test.values[:20],
    "Predicted": model_6.predict(X_test)[:20]
})

,Actual,Predicted
0,125.0,145.416885
1,121.0,131.141159
2,118.0,123.089966
3,111.0,116.330116
4,108.0,118.194908
5,107.0,116.225098
6,107.0,115.723663
7,107.0,113.070946
8,107.0,111.170128
9,108.0,108.398315


In [123]:
print("===== RANDOM FOREST =====")
evaluate_model(rf_models["6h"], X_test, y6_test, "RF 6h")
evaluate_model(rf_models["12h"], X_test, y12_test, "RF 12h")
evaluate_model(rf_models["24h"], X_test, y24_test, "RF 24h")

===== RANDOM FOREST =====


[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.4s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    2.7s
[Parallel(n_jobs=10)]: Done 200 out of 200 | elapsed:    3.1s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.


RF 6h
R2: 0.9704671958090201
RMSE: 6.784769085346224



[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.5s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    2.7s
[Parallel(n_jobs=10)]: Done 200 out of 200 | elapsed:    2.9s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.


RF 12h
R2: 0.8904385266925002
RMSE: 13.069152971921167



[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.5s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    3.0s
[Parallel(n_jobs=10)]: Done 200 out of 200 | elapsed:    3.2s finished


RF 24h
R2: 0.8199502354901002
RMSE: 16.760276635972904



In [124]:
print("===== XGBOOST =====")
evaluate_model(model_6, X_test, y6_test, "XGB 6h")
evaluate_model(model_12, X_test, y12_test, "XGB 12h")
evaluate_model(model_24, X_test, y24_test, "XGB 24h")

===== XGBOOST =====
XGB 6h
R2: 0.970813164862302
RMSE: 6.744911115104681

XGB 12h
R2: 0.9043127591234664
RMSE: 12.213651545519935

XGB 24h
R2: 0.7745511813432562
RMSE: 18.75464663062243



In [125]:
import pandas as pd

results = []

def collect_results(model, X_test, y_test, name):
    preds = model.predict(X_test)
    results.append({
        "Model": name,
        "R2": r2_score(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds))
    })

# RF
collect_results(rf_models["6h"], X_test, y6_test, "RF 6h")
collect_results(rf_models["12h"], X_test, y12_test, "RF 12h")
collect_results(rf_models["24h"], X_test, y24_test, "RF 24h")

# XGB
collect_results(model_6, X_test, y6_test, "XGB 6h")
collect_results(model_12, X_test, y12_test, "XGB 12h")
collect_results(model_24, X_test, y24_test, "XGB 24h")

pd.DataFrame(results)

[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.5s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    3.3s
[Parallel(n_jobs=10)]: Done 200 out of 200 | elapsed:    3.5s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.5s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    2.6s
[Parallel(n_jobs=10)]: Done 200 out of 200 | elapsed:    2.9s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    3.0s
[Parallel(n_jobs=10)]: Done 200 out of 200 | elapsed:    3.2s finished


,Model,R2,RMSE
0,RF 6h,0.970467,6.784769
1,RF 12h,0.890439,13.069153
2,RF 24h,0.819950,16.760277
3,XGB 6h,0.970813,6.744911
4,XGB 12h,0.904313,12.213652
5,XGB 24h,0.774551,18.754647


### Random Forest Regressor for 24 hours

In [128]:
rf_24 = rf_models["24h"]
joblib.dump(rf_24, "models/rf_24h.pkl")

['models/rf_24h.pkl']